In [12]:

import sklearn
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import importlib
import preproccessing_helper_functions 
importlib.reload(preproccessing_helper_functions)
from preproccessing_helper_functions import data_audit,rerunnable_insert



Importing in the dataset and doing a audit of the data 

In [13]:
raw_energy_dataset=pd.read_csv("energy_dataset.csv")
print("raw_energy_dataset statistics")
data_audit(raw_energy_dataset)


raw_energy_dataset statistics
number of rows: 35064
number of columns: 29
columns names:
 Index(['time', 'generation biomass', 'generation fossil brown coal/lignite',
       'generation fossil coal-derived gas', 'generation fossil gas',
       'generation fossil hard coal', 'generation fossil oil',
       'generation fossil oil shale', 'generation fossil peat',
       'generation geothermal', 'generation hydro pumped storage aggregated',
       'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation marine',
       'generation nuclear', 'generation other', 'generation other renewable',
       'generation solar', 'generation waste', 'generation wind offshore',
       'generation wind onshore', 'forecast solar day ahead',
       'forecast wind offshore eday ahead', 'forecast wind onshore day ahead',
       'total load forecast', 'total load actual', 'price day ahead',
       'price actual']

selecting feature 
didnt choose forecast wind eday ahead has it only contained NAN add more justification

In [14]:
filtered_energy_dataset=raw_energy_dataset[["time","forecast solar day ahead","forecast wind onshore day ahead",
       "total load forecast", "total load actual", "price day ahead","price actual"]].copy()
print("filtered_energy_dataset statistics")
data_audit(filtered_energy_dataset)

filtered_energy_dataset statistics
number of rows: 35064
number of columns: 7
columns names:
 Index(['time', 'forecast solar day ahead', 'forecast wind onshore day ahead',
       'total load forecast', 'total load actual', 'price day ahead',
       'price actual'],
      dtype='str')
percentage missing values by column:
total load actual                  0.001027
forecast solar day ahead           0.000000
time                               0.000000
forecast wind onshore day ahead    0.000000
total load forecast                0.000000
price day ahead                    0.000000
price actual                       0.000000
dtype: float64
missing values by column count:
time                                0
forecast solar day ahead            0
forecast wind onshore day ahead     0
total load forecast                 0
total load actual                  36
price day ahead                     0
price actual                        0
dtype: int64
dublicates:0

distinct values
forecast solar

auditing how time behaves when there is swapping between +1 and +2 for daylight savings 

In [ ]:

timeColTest=filtered_energy_dataset["time"].iloc[19556:19568]
print("As a string\n",timeColTest)
utcTest=pd.to_datetime(timeColTest,utc=True)
spainTest=utcTest.dt.tz_convert("Europe/Madrid")
print("utc\n",utcTest)
print("Europe/Madrid time zone\n",spainTest)

del timeColTest,utcTest,spainTest


As a string
 19556    2017-03-25 20:00:00+01:00
19557    2017-03-25 21:00:00+01:00
19558    2017-03-25 22:00:00+01:00
19559    2017-03-25 23:00:00+01:00
19560    2017-03-26 00:00:00+01:00
19561    2017-03-26 01:00:00+01:00
19562    2017-03-26 03:00:00+02:00
19563    2017-03-26 04:00:00+02:00
19564    2017-03-26 05:00:00+02:00
19565    2017-03-26 06:00:00+02:00
19566    2017-03-26 07:00:00+02:00
19567    2017-03-26 08:00:00+02:00
Name: time, dtype: str
utc
 19556   2017-03-25 19:00:00+00:00
19557   2017-03-25 20:00:00+00:00
19558   2017-03-25 21:00:00+00:00
19559   2017-03-25 22:00:00+00:00
19560   2017-03-25 23:00:00+00:00
19561   2017-03-26 00:00:00+00:00
19562   2017-03-26 01:00:00+00:00
19563   2017-03-26 02:00:00+00:00
19564   2017-03-26 03:00:00+00:00
19565   2017-03-26 04:00:00+00:00
19566   2017-03-26 05:00:00+00:00
19567   2017-03-26 06:00:00+00:00
Name: time, dtype: datetime64[us, UTC]
Europe/Madrid time zone
 19556   2017-03-25 20:00:00+01:00
19557   2017-03-25 21:00:00+01:00

After investiagting the output I will use the UTC as a consistant time index for chronologically ordering ,checking hourly intervals and linne up each oberservations with the target  24 hours ahead.Madrid is used as for creating local calender features such as hour of the day ,weekend and holidays.

In [16]:
filtered_energy_dataset=filtered_energy_dataset.rename(columns={"time":"time_utc"}).copy()
filtered_energy_dataset["time_utc"]=pd.to_datetime(filtered_energy_dataset["time_utc"],utc=True)
print(filtered_energy_dataset.columns)

rerunnable_insert(filtered_energy_dataset,filtered_energy_dataset.columns.tolist().index("time_utc") + 1,"time_Madrid",filtered_energy_dataset["time_utc"].dt.tz_convert("Europe/Madrid"))


print(filtered_energy_dataset.columns)
print(filtered_energy_dataset[["time_utc","time_Madrid"]].iloc[19556:19568])



Index(['time_utc', 'forecast solar day ahead',
       'forecast wind onshore day ahead', 'total load forecast',
       'total load actual', 'price day ahead', 'price actual'],
      dtype='str')
Index(['time_utc', 'time_Madrid', 'forecast solar day ahead',
       'forecast wind onshore day ahead', 'total load forecast',
       'total load actual', 'price day ahead', 'price actual'],
      dtype='str')
                       time_utc               time_Madrid
19556 2017-03-25 19:00:00+00:00 2017-03-25 20:00:00+01:00
19557 2017-03-25 20:00:00+00:00 2017-03-25 21:00:00+01:00
19558 2017-03-25 21:00:00+00:00 2017-03-25 22:00:00+01:00
19559 2017-03-25 22:00:00+00:00 2017-03-25 23:00:00+01:00
19560 2017-03-25 23:00:00+00:00 2017-03-26 00:00:00+01:00
19561 2017-03-26 00:00:00+00:00 2017-03-26 01:00:00+01:00
19562 2017-03-26 01:00:00+00:00 2017-03-26 03:00:00+02:00
19563 2017-03-26 02:00:00+00:00 2017-03-26 04:00:00+02:00
19564 2017-03-26 03:00:00+00:00 2017-03-26 05:00:00+02:00
19565 2017-03-2

preprocessing 
time has to be converted to time value potentially 
total load actual has 36 empty values how to handle that 

Baseline model implemenation 